In [ ]:
from pm4py.objects.log.importer.xes import factory as xes_importer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np


In [ ]:
def log_to_sequences(log):
    traces = []
    for trace in log:
        traces.append([e["concept:name"] for e in trace])
    return traces


In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sequences)  
vocab_size = len(tokenizer.word_index) + 1


In [ ]:
def build_X_y(sequences, tokenizer):
    X = []
    y = []

    for seq in sequences:
        encoded = tokenizer.texts_to_sequences([seq])[0]

        for i in range(1, len(encoded)):
            X.append(encoded[:i])
            y.append(encoded[i])

    max_len = max(len(x) for x in X)
    X = pad_sequences(X, maxlen=max_len, padding='pre')
    y = np.array(y)

    return X, y, max_len


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

def build_lstm_model(vocab, max_len):
    model = Sequential()
    model.add(Embedding(vocab, 64, input_length=max_len))
    model.add(LSTM(128, return_sequences=False))
    model.add(Dense(vocab, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    return model


In [ ]:
X, y, max_len = build_X_y(sequences, tokenizer)
model = build_lstm_model(vocab_size, max_len)

model.fit(X, y, epochs=10, batch_size=32)


In [ ]:
import math

def score_trace(model, tokenizer, max_len, trace):
    events = tokenizer.texts_to_sequences([trace])[0]
    score = 0.0
    count = 0

    for i in range(1, len(events)):
        X = pad_sequences([events[:i]], maxlen=max_len, padding='pre')
        y_true = events[i]

        prob = model.predict(X, verbose=0)[0][y_true]

        # Lower probability → more anomalous
        score += -math.log(prob + 1e-9)
        count += 1

    return score / count  # lower = normal, higher = anomaly


In [ ]:
def find_anomalies_lstm(model, tokenizer, max_len, test_sequences, threshold=None):
    scores = []

    for trace in test_sequences:
        s = score_trace(model, tokenizer, max_len, trace)
        scores.append(s)

    if threshold is None:
        # Standard threshold: mean + 3*std
        threshold = np.mean(scores) + 3*np.std(scores)

    anomaly_flags = [s > threshold for s in scores]

    return scores, anomaly_flags
